# 🔍 Day 02a: DNS — How Domain Names Become IP Addresses

---

## 🎯 What You'll Master Today
- DNS hierarchy and zone delegation
- Recursive vs iterative resolution
- DNS record types (A, AAAA, CNAME, MX, NS, TXT)
- DNS caching and TTL

---

### 🎭 The Analogy

DNS = the internet's phonebook. You know the name (example.com), DNS gives you the number (93.184.216.34).

```
╔══════════════════════════════════════════════════════════════════════╗
║                    DNS HIERARCHY                                     ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║                          . (Root)                                    ║
║                       /    |     \                                   ║
║                    .com   .org   .in                                 ║
║                   / | \                                              ║
║            google  amazon  example                                   ║
║             / \                                                      ║
║          www  mail                                                   ║
║                                                                      ║
║  FQDN: www.google.com.  (note the trailing dot = root)             ║
║                                                                      ║
║  Root servers:  13 logical (A-M), ~1500 physical (anycast)          ║
║  TLD servers:   .com, .org, .net, .in — managed by registries       ║
║  Authoritative: The final answer for a domain (ns1.google.com)     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. DNS Resolution Simulator
# ============================================

class DNSServer:
    def __init__(self, name, records=None, referrals=None):
        self.name = name
        self.records = records or {}
        self.referrals = referrals or {}
    
    def query(self, domain, qtype="A"):
        key = (domain, qtype)
        if key in self.records:
            return {"type": "answer", "value": self.records[key]}
        # Check for referral
        for suffix, ns in self.referrals.items():
            if domain.endswith(suffix):
                return {"type": "referral", "ns": ns}
        return {"type": "nxdomain"}

# Build DNS hierarchy
root = DNSServer("Root (.)", referrals={
    ".com": "TLD-COM-Server",
    ".org": "TLD-ORG-Server",
})

tld_com = DNSServer("TLD (.com)", referrals={
    "example.com": "ns1.example.com",
    "google.com": "ns1.google.com",
})

auth_example = DNSServer("ns1.example.com", records={
    ("example.com", "A"): "93.184.216.34",
    ("www.example.com", "A"): "93.184.216.34",
    ("example.com", "MX"): "10 mail.example.com",
    ("example.com", "NS"): "ns1.example.com",
    ("mail.example.com", "A"): "93.184.216.35",
})

servers = {
    "Root": root,
    "TLD-COM-Server": tld_com,
    "ns1.example.com": auth_example,
}

def iterative_resolve(domain, qtype="A"):
    """Simulate iterative DNS resolution."""
    print(f"  ═══ Resolving: {domain} ({qtype}) ═══\n")
    
    current_server = "Root"
    step = 1
    
    while True:
        server = servers[current_server]
        result = server.query(domain, qtype)
        
        if result["type"] == "answer":
            print(f"    Step {step}: Ask {server.name}")
            print(f"           → ANSWER: {result['value']}")
            print(f"\n  ✅ Resolved: {domain} → {result['value']}")
            return result["value"]
        elif result["type"] == "referral":
            print(f"    Step {step}: Ask {server.name}")
            print(f"           → REFERRAL: Try {result['ns']}")
            current_server = result["ns"]
        else:
            print(f"    Step {step}: Ask {server.name} → NXDOMAIN")
            return None
        step += 1

iterative_resolve("www.example.com")
print()
iterative_resolve("example.com", "MX")

In [ ]:
# ============================================
# 2. Recursive vs Iterative Resolution
# ============================================

print("""
  ═══ RECURSIVE vs ITERATIVE ═══

  RECURSIVE (what your browser does):
  ┌────────┐    ┌──────────────┐    ┌──────┐    ┌─────┐    ┌──────┐
  │Browser │ →  │ ISP Resolver │ →  │ Root │ →  │ TLD │ →  │ Auth │
  │        │ ←  │  (does all   │ ←  │      │ ←  │     │ ←  │      │
  └────────┘    │  the work)   │    └──────┘    └─────┘    └──────┘
                └──────────────┘
  Client asks ONCE. Resolver chases referrals.

  ITERATIVE (what the resolver does internally):
  ┌──────────────┐
  │   Resolver   │ → Root: "Ask .com TLD"               (referral)
  │              │ → TLD:  "Ask ns1.example.com"         (referral)
  │              │ → Auth: "93.184.216.34"               (answer!)
  └──────────────┘
  Resolver asks each server directly. Server returns referral OR answer.

  Key difference:
  - Recursive: "Please get this for me" (burden on resolver)
  - Iterative: "I'll ask you, then ask the next" (resolver does work)
  - Your stub resolver (browser/OS) does recursive
  - DNS servers do iterative between themselves
""")

In [ ]:
# ============================================
# 3. DNS Record Types
# ============================================

records = [
    ("A",     "IPv4 address",              "example.com → 93.184.216.34"),
    ("AAAA",  "IPv6 address",              "example.com → 2606:2800:220:1:..."),
    ("CNAME", "Canonical name (alias)",    "www.example.com → example.com"),
    ("MX",    "Mail exchange",             "example.com → 10 mail.example.com"),
    ("NS",    "Name server",              "example.com → ns1.example.com"),
    ("TXT",   "Text (SPF, DKIM, verify)", "example.com → v=spf1 include:..."),
    ("SOA",   "Start of authority",       "Zone metadata (serial, refresh, retry)"),
    ("PTR",   "Reverse DNS (IP→name)",    "34.216.184.93 → example.com"),
    ("SRV",   "Service locator",          "_sip._tcp.example.com → host:port"),
]

print("  ╔══════════════════════════════════════════════════════════════════╗")
print("  ║                    DNS RECORD TYPES                             ║")
print("  ╠══════════════════════════════════════════════════════════════════╣")
for rtype, desc, example in records:
    print(f"  ║  {rtype:5} │ {desc:28} │ {example:30} ║")
print("  ╚══════════════════════════════════════════════════════════════════╝")

print("\n  💼 CNAME gotchas (interview favorite):")
print("    - CNAME = alias to another domain name")
print("    - Cannot coexist with other records at same name")
print("    - Cannot be at zone apex (example.com) — use ALIAS/ANAME instead")
print("    - Adds extra DNS lookup (CNAME → A record)")

In [ ]:
# ============================================
# 4. DNS Caching & TTL
# ============================================

import time

class DNSCache:
    def __init__(self):
        self.cache = {}  # domain → (ip, expire_time)
        self.hits = 0
        self.misses = 0
    
    def put(self, domain, ip, ttl):
        self.cache[domain] = (ip, time.time() + ttl)
    
    def get(self, domain):
        if domain in self.cache:
            ip, expire = self.cache[domain]
            if time.time() < expire:
                self.hits += 1
                remaining = int(expire - time.time())
                return ip, remaining
            else:
                del self.cache[domain]
        self.misses += 1
        return None, 0

cache = DNSCache()

# Simulate caching
print("  ═══ DNS Caching Demo ═══\n")

cache.put("example.com", "93.184.216.34", ttl=300)
cache.put("google.com", "142.250.80.46", ttl=60)

# Query cached
ip, ttl_left = cache.get("example.com")
print(f"  Cache HIT:  example.com → {ip} (TTL: {ttl_left}s remaining)")

ip, ttl_left = cache.get("unknown.com")
print(f"  Cache MISS: unknown.com → {ip} (full resolution needed)")

print(f"\n  Hits: {cache.hits}, Misses: {cache.misses}")

print("""
  DNS Caching layers:
    1. Browser cache          (Chrome: chrome://net-internals/#dns)
    2. OS cache               (Windows: ipconfig /displaydns)
    3. Router cache
    4. ISP recursive resolver
    5. Authoritative server (provides TTL)

  TTL values:
    Low  (60s):   Frequently changing (load balancer, failover)
    Med  (300s):  Normal websites
    High (86400s): Stable infrastructure (1 day)
""")

In [ ]:
# ============================================
# 5. "What Happens When You Type a URL?"
# ============================================

print("""
  ═══ INTERVIEW CLASSIC: What happens when you type google.com? ═══

   1. Browser checks URL format (scheme, host, path)
   2. Browser checks HSTS list (force HTTPS?)
   3. DNS Resolution:
      a. Browser cache → OS cache → Router → ISP resolver
      b. If miss: Root → .com TLD → google's auth NS → IP
   4. TCP connection: 3-way handshake (SYN → SYN-ACK → ACK)
   5. TLS handshake: ClientHello → ServerHello → cert verify → keys
   6. HTTP request: GET / HTTP/2 (with headers)
   7. Server processes: load balancer → web server → app → DB
   8. HTTP response: 200 OK + HTML body
   9. Browser parses HTML → DOM tree
  10. Browser fetches CSS, JS, images (more HTTP requests)
  11. CSSOM + DOM → Render tree → Layout → Paint → Composite
  12. JavaScript executes (can modify DOM)
  13. Page is interactive!

  Total time: ~100ms - 3s (depending on network, server, complexity)
""")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What happens when you type google.com? | DNS → TCP → TLS → HTTP → Parse → Render (full pipeline) |
| 2 | Recursive vs iterative DNS? | Recursive: resolver does all work. Iterative: resolver follows referrals itself |
| 3 | What is a CNAME record? | Alias pointing to another domain. Cannot be at zone apex. Adds extra lookup |
| 4 | What is DNS TTL? | Time-to-live: how long to cache. Low=flexibility, High=performance |
| 5 | Why 13 root servers? | Fits in one 512-byte UDP packet. Actually 1500+ physical servers via anycast |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • DNS: domain → IP (the internet's phonebook)          │
## │  • Hierarchy: Root → TLD → Authoritative                 │
## │  • Records: A(IPv4), AAAA(IPv6), CNAME, MX, NS, TXT    │
## │  • Caching at every level (browser, OS, ISP)             │
## │  • "What happens when..." = #1 CN interview question    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Email Protocols** — SMTP, POP3, IMAP